# Part 2 — Saliency Maps: Where Does the CNN Look?

We compute the gradient of each target variable prediction with respect to the full
450×449 input map.  The absolute gradient `|∂ŷ_v / ∂X|` is our **saliency map**: pixels
with large gradients are the geographic locations whose weather state most strongly
influences the 24-hour forecast at the Jumbo Statue (Tufts University).

We average over **150 random samples per meteorological season** from **2022** (a year
the model never trained on), giving four stable, season-specific sensitivity maps.

**Checkpoint**: swap `CHECKPOINT_PATH` below to point to any trained `best.pt`.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Make sure the repo root is on the path so we can import subhanga.*
REPO_ROOT = Path('/cluster/tufts/c26sp1cs0137/supadh03/SkyOracle')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from subhanga.models import WeatherResNetGAP
from subhanga.data   import DatasetPaths, _timestamp_to_input_path

print('Imports OK')

In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────
# Swap this path to use a different checkpoint (e.g. new best.pt after job finishes)
CHECKPOINT_PATH = REPO_ROOT / 'checkpoints/subhanga/best.pt'

DATASET_DIR     = Path('/cluster/tufts/c26sp1cs0137/data/assignment2_data/dataset')
FIGURES_DIR     = REPO_ROOT / 'subhanga/figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Year(s) to use for saliency — held-out, never seen during training (2018-2020) or val (2021)
SALIENCY_YEARS  = [2022]

# Samples per season (higher = smoother maps but slower)
N_SAMPLES       = 150

DEVICE          = 'cpu'   # CPU is fine for gradient computation in notebooks
SEED            = 137

# Jumbo Statue WGS-84 coordinates (for plotting)
JUMBO_LAT = 42.40777867717294
JUMBO_LON = -71.12041637590173

# Target variable display names (matches order in targets.pt → values columns)
TARGET_NAMES = [
    'TMP@2m (K)',
    'RH@2m (%)',
    'UGRD@10m (m/s)',
    'VGRD@10m (m/s)',
    'GUST@surface (m/s)',
    'APCP@surface (mm)',
]

print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Dataset    : {DATASET_DIR}')
print(f'Figures    : {FIGURES_DIR}')

In [ ]:
# ─── Load checkpoint and reconstruct model ───────────────────────────────────
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
cfg  = ckpt['config']

model = WeatherResNetGAP(
    in_channels  = 42,
    base_channels= cfg['base_channels'],
    out_dim      = 6,
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
model.to(DEVICE)

channel_mean = ckpt.get('channel_mean')  # (42,) or None
channel_std  = ckpt.get('channel_std')   # (42,) or None

if channel_mean is not None:
    channel_mean = channel_mean.float().to(DEVICE)
    channel_std  = channel_std.float().to(DEVICE)

print(f"Loaded epoch {ckpt['epoch']}  |  base_channels={cfg['base_channels']}")
print(f"Channel normalisation: {'ON' if channel_mean is not None else 'OFF'}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

In [ ]:
# ─── Load metadata: grid coordinates and Jumbo Statue indices ─────────────────
meta     = torch.load(DATASET_DIR / 'metadata.pt', weights_only=False)
x_coords = np.asarray(meta['grid_x'])  # (449,)  Lambert-Conformal x in metres
y_coords = np.asarray(meta['grid_y'])  # (450,)  Lambert-Conformal y in metres
jumbo_y  = int(meta['jumbo_y_idx'])    # 177
jumbo_x  = int(meta['jumbo_x_idx'])    # 263
H, W     = len(y_coords), len(x_coords)

# Reconstruct the Lambert Conformal projection used by the HRRR NE grid
PROJECTION = ccrs.LambertConformal(
    central_longitude   = 262.5,
    central_latitude    = 38.5,
    standard_parallels  = (38.5, 38.5),
    globe = ccrs.Globe(semimajor_axis=6371229, semiminor_axis=6371229),
)

print(f'Grid        : {H} × {W}  (rows × cols)')
print(f'Jumbo index : y={jumbo_y}, x={jumbo_x}')

In [ ]:
# ─── Load targets and build per-season sample index lists ────────────────────
targets   = torch.load(DATASET_DIR / 'targets.pt', weights_only=False)
all_times = np.asarray(targets['time'])         # datetime64
y_reg     = targets['values'].float()            # (T, 6)
T         = len(all_times)

pd_times  = pd.DatetimeIndex(all_times)

# NaN-valid target mask  (t is usable only if target at t+24 is not NaN)
valid_target = (~y_reg.isnan().any(dim=1)).numpy()  # (T,)

# Meteorological seasons (month sets)
SEASONS = {
    'Winter': [12, 1, 2],
    'Spring': [3, 4, 5],
    'Summer': [6, 7, 8],
    'Fall'  : [9, 10, 11],
}

def get_season_indices(years, season_months):
    """Return all t-indices in the given years & season that have valid targets."""
    year_arr  = pd_times.year
    month_arr = pd_times.month
    mask = (
        np.isin(year_arr,  years)         # correct year
        & np.isin(month_arr, season_months)  # correct season
        & valid_target                       # target at t is valid
    )
    t_all   = np.where(mask)[0]
    # Also need target at t+24 to exist (t+24 < T) and that row to be non-NaN
    t_valid = t_all[(t_all + 24 < T) & valid_target[t_all + 24]]
    return t_valid

rng = np.random.default_rng(SEED)

season_indices = {}
for name, months in SEASONS.items():
    t_all = get_season_indices(SALIENCY_YEARS, months)
    chosen = rng.choice(t_all, min(N_SAMPLES, len(t_all)), replace=False)
    season_indices[name] = np.sort(chosen)
    print(f'{name:6s}: {len(t_all):4d} valid  →  sampling {len(chosen)}')

In [ ]:
# ─── Helper functions ─────────────────────────────────────────────────────────
paths = DatasetPaths(dataset_dir=DATASET_DIR)

def load_and_normalise(t_idx: int) -> torch.Tensor:
    """Load input at time t_idx, fill NaNs, apply channel normalisation.
    Returns (C, H, W) float32 on DEVICE."""
    ts     = all_times[t_idx]
    x_path = _timestamp_to_input_path(paths.inputs_dir, ts)
    x = torch.load(x_path, weights_only=True).float()  # (H, W, C)
    x = x.permute(2, 0, 1)                              # (C, H, W)
    x = torch.nan_to_num(x, nan=0.0)                    # fill all NaNs
    if channel_mean is not None:
        x = (x - channel_mean[:, None, None]) / channel_std[:, None, None]
    return x.to(DEVICE)


def compute_saliency(x: torch.Tensor) -> np.ndarray:
    """Compute saliency maps for all 6 target variables for one input sample.

    Args
        x: (C, H, W) normalised input tensor
    Returns
        sals: (6, H, W) numpy array of |∂ŷ_v / ∂X| (max over channels)
    """
    # Detach and add batch dim, then enable gradient tracking on the input
    xb = x.detach().unsqueeze(0).requires_grad_(True)  # (1, C, H, W)

    with torch.set_grad_enabled(True):
        pred = model(xb)  # (1, 6) — normalised predictions

        sals = np.empty((6, H, W), dtype=np.float32)
        for v in range(6):
            if xb.grad is not None:
                xb.grad.zero_()
            # retain_graph so we can back-prop for each variable separately
            pred[0, v].backward(retain_graph=(v < 5))
            # Aggregate channel dimension: take max absolute gradient over all 42 channels
            sals[v] = xb.grad[0].abs().max(dim=0)[0].cpu().numpy()  # (H, W)

    return sals

print('Helpers defined.')

In [ ]:
# ─── Compute and average saliency maps per season ────────────────────────────
# Result: seasonal_saliency[season] = (6, H, W) mean-absolute-gradient map

seasonal_saliency = {}

for season_name, t_indices in season_indices.items():
    accum   = np.zeros((6, H, W), dtype=np.float64)
    n_ok    = 0
    n_skip  = 0

    for t_idx in t_indices:
        try:
            x    = load_and_normalise(int(t_idx))
            sals = compute_saliency(x)
            accum += sals
            n_ok  += 1
        except FileNotFoundError:
            n_skip += 1
            continue

    mean_sal = (accum / n_ok).astype(np.float32) if n_ok > 0 else accum.astype(np.float32)
    seasonal_saliency[season_name] = mean_sal
    print(f'{season_name:6s}: {n_ok} OK, {n_skip} skipped')

print('\nAll seasonal saliency maps computed.')

In [ ]:
# ─── Persist saliency maps so you can re-run plots without recomputing ────────
save_path = FIGURES_DIR / 'seasonal_saliency.npz'
np.savez(save_path, **{k: v for k, v in seasonal_saliency.items()})
print(f'Saliency arrays saved → {save_path}')

# To reload later:  data = np.load(save_path);  seasonal_saliency = dict(data)

In [ ]:
# ─── Figure 1: Combined seasonal saliency (2×2 grid) ─────────────────────────
# One panel per season.  Saliency = max over all 6 target variables.
# This is the main figure for the report / slide deck.

SEASON_COLORS = {
    'Winter': '#1f77b4',
    'Spring': '#2ca02c',
    'Summer': '#d62728',
    'Fall'  : '#ff7f0e',
}

fig, axes = plt.subplots(
    2, 2,
    figsize=(18, 14),
    subplot_kw={'projection': PROJECTION},
)

for ax, (season_name, sal_6xHxW) in zip(axes.flat, seasonal_saliency.items()):

    # Combine across variables: max sensitivity at each grid point
    combined = sal_6xHxW.max(axis=0)  # (H, W)

    # Clip to 99th percentile to suppress isolated spikes
    vmax = float(np.percentile(combined, 99))

    mesh = ax.pcolormesh(
        x_coords, y_coords, combined,
        transform = PROJECTION,
        cmap      = 'hot_r',
        vmin      = 0,
        vmax      = vmax,
        shading   = 'auto',
    )

    # Geographic overlays
    ax.add_feature(cfeature.COASTLINE,  linewidth=0.9)
    ax.add_feature(cfeature.STATES,     linewidth=0.5, edgecolor='white')
    ax.add_feature(cfeature.BORDERS,    linewidth=0.7, linestyle='--', edgecolor='white')
    ax.add_feature(cfeature.LAKES,      facecolor='#4a9aba', alpha=0.5)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
    gl.top_labels   = False
    gl.right_labels = False

    # Mark Jumbo Statue
    ax.plot(
        JUMBO_LON, JUMBO_LAT,
        marker='*', color='cyan', markersize=16, markeredgecolor='black', markeredgewidth=0.5,
        transform=ccrs.PlateCarree(), zorder=6, label='Jumbo Statue'
    )
    ax.legend(loc='lower right', fontsize=9)

    cbar = plt.colorbar(mesh, ax=ax, fraction=0.035, pad=0.04)
    cbar.set_label('|∂ŷ / ∂X|  (max over variables & channels)', fontsize=9)

    ax.set_title(
        f'{season_name}',
        fontsize=15, fontweight='bold',
        color=SEASON_COLORS[season_name],
    )

fig.suptitle(
    'Saliency Maps — Which regions drive the 24h forecast at Tufts?\n'
    f'(Average over {N_SAMPLES} samples per season from {SALIENCY_YEARS[0]}, '
    f'model: epoch {ckpt["epoch"]}, base_channels={cfg["base_channels"]})',
    fontsize=14, y=1.01,
)

plt.tight_layout()
out_path = FIGURES_DIR / 'fig_saliency_seasonal_combined.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

In [ ]:
# ─── Figure 2: Per-variable saliency — Winter (strongest upstream signal) ─────
# 2-row × 3-col subplot, one panel per output variable.

def plot_per_variable(season_name, cmap='plasma', save_suffix=''):
    sal_6xHxW = seasonal_saliency[season_name]

    fig, axes = plt.subplots(
        2, 3,
        figsize=(22, 14),
        subplot_kw={'projection': PROJECTION},
    )

    for ax, var_idx, var_name in zip(axes.flat, range(6), TARGET_NAMES):
        sal  = sal_6xHxW[var_idx]                     # (H, W)
        vmax = float(np.percentile(sal, 99))

        mesh = ax.pcolormesh(
            x_coords, y_coords, sal,
            transform = PROJECTION,
            cmap      = cmap,
            vmin      = 0,
            vmax      = vmax,
            shading   = 'auto',
        )

        ax.add_feature(cfeature.COASTLINE, linewidth=0.9)
        ax.add_feature(cfeature.STATES,    linewidth=0.5, edgecolor='white')
        ax.add_feature(cfeature.BORDERS,   linewidth=0.7, linestyle='--', edgecolor='white')
        ax.add_feature(cfeature.LAKES,     facecolor='#4a9aba', alpha=0.5)

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
        gl.top_labels   = False
        gl.right_labels = False

        ax.plot(
            JUMBO_LON, JUMBO_LAT,
            marker='*', color='cyan', markersize=16,
            markeredgecolor='black', markeredgewidth=0.5,
            transform=ccrs.PlateCarree(), zorder=6,
        )

        cbar = plt.colorbar(mesh, ax=ax, fraction=0.035, pad=0.04)
        cbar.set_label('|∂ŷ / ∂X|', fontsize=9)
        ax.set_title(f'∂({var_name}) / ∂X', fontsize=11)

    fig.suptitle(
        f'{season_name} — Per-Variable Saliency Maps\n'
        f'(Average over {N_SAMPLES} samples, {SALIENCY_YEARS[0]})',
        fontsize=14, y=1.01,
    )
    plt.tight_layout()
    suffix = save_suffix or season_name.lower()
    out_path = FIGURES_DIR / f'fig_saliency_{suffix}_per_variable.png'
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    print(f'Saved → {out_path}')
    plt.show()


plot_per_variable('Winter', cmap='plasma')

In [ ]:
# ─── Figure 3: Per-variable saliency — Summer (convective regime, different pattern) ─
plot_per_variable('Summer', cmap='plasma')

In [ ]:
# ─── Figure 4: Temperature sensitivity across all four seasons side-by-side ───
# Shows how the "catchment area" for 2m temperature shifts seasonally.

VAR_IDX   = 0  # TMP@2m
VAR_LABEL = TARGET_NAMES[VAR_IDX]

fig, axes = plt.subplots(
    1, 4,
    figsize=(28, 7),
    subplot_kw={'projection': PROJECTION},
)

# Use a shared colour scale across seasons for direct comparison
all_vals  = np.concatenate([s[VAR_IDX].ravel() for s in seasonal_saliency.values()])
vmax_glob = float(np.percentile(all_vals, 99))

for ax, (season_name, sal_6xHxW) in zip(axes, seasonal_saliency.items()):
    sal  = sal_6xHxW[VAR_IDX]  # (H, W)

    mesh = ax.pcolormesh(
        x_coords, y_coords, sal,
        transform = PROJECTION,
        cmap      = 'hot_r',
        vmin      = 0,
        vmax      = vmax_glob,
        shading   = 'auto',
    )

    ax.add_feature(cfeature.COASTLINE, linewidth=0.9)
    ax.add_feature(cfeature.STATES,    linewidth=0.5, edgecolor='white')
    ax.add_feature(cfeature.BORDERS,   linewidth=0.7, linestyle='--', edgecolor='white')
    ax.add_feature(cfeature.LAKES,     facecolor='#4a9aba', alpha=0.5)

    ax.plot(
        JUMBO_LON, JUMBO_LAT,
        marker='*', color='cyan', markersize=16,
        markeredgecolor='black', markeredgewidth=0.5,
        transform=ccrs.PlateCarree(), zorder=6,
    )

    ax.set_title(season_name, fontsize=13, fontweight='bold',
                 color=SEASON_COLORS[season_name])

# Single shared colorbar on the right
fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.015, 0.7])
sm  = plt.cm.ScalarMappable(cmap='hot_r', norm=mcolors.Normalize(vmin=0, vmax=vmax_glob))
sm.set_array([])
fig.colorbar(sm, cax=cax, label=f'|∂({VAR_LABEL}) / ∂X|')

fig.suptitle(
    f'Seasonal shift in {VAR_LABEL} sensitivity — shared colour scale\n'
    '(upstream catchment area of the forecast moves with the jet stream)',
    fontsize=13, y=1.03,
)

out_path = FIGURES_DIR / 'fig_saliency_tmp_seasonal_comparison.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

## Interpretation Guide for the Report

### What to look for in Figure 1 (combined seasonal maps)
- **Upstream hot-spot**: The region of highest saliency should sit **northwest of Boston**.
  At 3 km/pixel and ~800 km/day storm advection, a 24 h ahead signal travels roughly
  **267 pixels** upstream — placing it over upstate New York, Quebec, or the Great Lakes.
- **Seasonal shift**: In winter the jet stream sits further south, so storm tracks enter
  New England from the southwest (mid-Atlantic corridor).  In summer, Canadian cold
  fronts and the Bermuda High suppress that track; sensitivity should be more diffuse.
- **Ocean vs. land**: Large saliency over the Atlantic Ocean east of Boston would suggest
  the model learned a spurious correlation (weather there arrives *after* Boston).

### What to look for in Figures 2 & 3 (per-variable maps)
- **TMP@2m and GUST**: Broad upstream sensitivity — temperature and wind are dominated
  by large-scale advection from hundreds of km away.
- **APCP (precipitation)**: More localised sensitivity — precipitation events are often
  tied to mesoscale systems (10–100 km) and local moisture convergence.
- **UGRD / VGRD (wind components)**: Expect sensitivity along the wind-shear zones
  (often the coastline and the ridge of the Appalachians).

### What to look for in Figure 4 (seasonal TMP comparison, shared scale)
- The **centroid of the hot-spot** should migrate seasonally.  You can report the
  approximate lat/lon of the centroid for each season as a quantitative finding.
- If the maps look nearly identical across seasons, the model may not have enough
  capacity to capture season-dependent dynamics — worth mentioning as a limitation.

### Report / presentation framing
> *"Our saliency analysis reveals that the CNN has implicitly learned the concept of
> atmospheric advection: its predictions are most sensitive to weather conditions
> located ~200–300 pixels (600–900 km) northwest of the forecast target, consistent
> with the mid-latitude westerly wind belt.  This physical coherence gives us confidence
> that the model is exploiting real meteorological signals rather than statistical artefacts."*